#### The building blocks of CNNs

**Understanding CNNs and Feature Hierarchies**
- Successfully extracting salien features is key to the performance of any ML algorithm.
- Certain types of NNs, such as CNNs, can automatically learn the features from raw data that are most useful for a particular task.
- It's common to cosinder CNNs as feature extractors:\
(i) the early layers (CNN) extract *low-level features* from raw data.\
(ii) the later layers (DNN) use these features to *make a prediction* on either a continuous target value or class label.
- Deep CNNs construct a so-called *feature hierarchy* by combing the low-level features in a layer-wise fashion to form high-level features.\
(i) In images, *low-level features* are edges, blobs,... are extracted from the *early layers*.\
(ii) The *low-level features* are then combined to form *high-level features*.\
(iii) Those *high-level features* can form more complex sahpes, such as buildings, cats, dogs.

<img src='images/feature_map.png' width=400>

- CNNs: images -> feature map: a local patch of pixels -> an element in feature map.
- Local patch of pixes ~ local receptive field.
- Hence, CNNs perform very well on image-related tasks because of two important ideas:\
(i) Sparse connectivity: a single elemen ~ a small patch of pixes -> local attention -> improve the ability to capture salient features from nearby pixels.\
(ii) Parameter sharing: the same weights are used for different patches of the input image of the same layer ~ efficient.
- CNNs are often composed of:\
(i) convolutional layers\
(ii) subsampling layers (pooling layers)\
(iii) fully connected layers

**Convolution vs dot product**
- x: input or signal
- w: filter or kernel
- s: stride ~ shifting the kernel on input
- convolution ~ cross-correlation in CNN (dot product with rotated w and shifted it along x).

<img src='images/convolution_vs_dot_product.png' width=600>

**Padding to control the size of the output feature maps**
- The padding is added to control the size of output, and also, how the boundary cells may be treated.
- There are 3 modes of padding that are commonly used:\
(i) full: output size > input size (if s=1, k=3 -> add 2 more cells in each end | all cells are treated the same) -> rarely used\
(ii) same: output size = input size (if s=1, k=3 -> add 1 more cells in each end | edge cells has more computations but still less than other inner cells) -> most common used\
(iii) valid: output size < input size (no addition of new cell | edge cells has less computation than inner cells)

<img src='images/padding.png' width=600>

**Determining the size of the convolution output**
- p: padding
- n: size of input vector
- m: size of kernel/filter
- s: stride
- o: output size

o = floor((2p + n - m)/s) + 1

In [5]:
import numpy as np 
n = 10
m = 5
s = 1

# full padding
p = 2
o_full = np.floor((2*p + n - m)/s) + 1
print(o_full)

# same padding
p = 1
o_same = np.floor((2*p + n -m)/s) + 1
print(o_same)

# valid padding
p = 0
o_valid = np.floor((2*p + n - m)/s) + 1
print(o_valid)

10.0
8.0
6.0


In [20]:
# check convoluation ~ cross-correlation with rotated w
x = [1, 3, 2, 4, 5, 6, 1]
w = [1, 0, 3]

print("Convolve: ", np.convolve(x, w, mode="same"))
w_rotated = w[::-1]
print("Cross correlation with rotated w: ", np.correlate(x, w_rotated, mode="same"))

Convolve:  [ 3  5 13 11 18 16 18]
Cross correlation with rotated w:  [ 3  5 13 11 18 16 18]


**2D Convolution**

<img src='images/2d_convolution.png' width=500>

- For example: p = (1,1), s =(2,2,), X=(3,3), w=(3,3)

<img src='images/2d_computation.png' width=500>

In [86]:
# X
X = np.array([
    [2, 1, 2],
    [5, 0, 1],
    [1, 7, 3]
])

# W
W = np.array([
    [0.5, 0.7, 0.4],
    [0.3, 0.4, 0.1],
    [0.5, 1, 0.5]
])

# W rotated
W_rotated = np.flip(W)
print(W_rotated)

# X padded
X_padded = np.pad(X, pad_width=1, mode="constant", constant_values=0)
print(X_padded)

stride = 2
padding = 1
x_size = 3
w_size = 3

output_size = np.int8((np.floor((2*1 + x_size - w_size) / stride) + 1))
print(output_size)

output = np.zeros((output_size, output_size))

# X * W
row = 0
for i in range(len(X)):
    col = 0

    if i % stride != 0:
        continue

    for j in range(len(X[0])):
        if j % stride != 0:
            continue
        val = X_padded[i:i+3, j:j+3] * W_rotated
        output[row, col] = np.sum(val)
        col += 1
    row += 1

print(output)

[[0.5 1.  0.5]
 [0.1 0.4 0.3]
 [0.4 0.7 0.5]]
[[0 0 0 0 0]
 [0 2 1 2 0]
 [0 5 0 1 0]
 [0 1 7 3 0]
 [0 0 0 0 0]]
2
[[4.6 1.6]
 [7.5 2.9]]


**Subsampling layers**
- Typically in 2 forms of pooling operation in CNNs: max-pooling and mean-pooling (average-pooling)

<img src='images/pooling.png' width=500>

- Max-Pooling introduces a local invariance -> small changes in a local neighborhood do not change the result of max-pooling -> robust to noise in the input data.
- Pooling decreases the size of features -> higher computational efficiency. Also, reducing the number of features may reduce the degree of overfitting as well.
- In term of feature deduction, Pooling (2,2) = convolution layer with stride = 2, but Convolution approach does have learnable weights.

**Trainable param count**

<img src='images/params_count.png' width=600>

kernel = m1xm2
input_channel = c_in
output_channel = c_out

num_params = c_in * c_out * m1 * m2 + c_out

*In case of fully connected layer: (assume same hidden units):*\
 (c_in * n1 * n2) * (c_out * n1 * n2) + (c_out * n1 * n2) = c_in * c_out * (n1 * n2)^2 + (c_out * n1 * n2)

**Regularizers**
- Can apply L1 or L2 by extending the loss function.
- L2 can be applied under optimizer as weight_decay in AdamW or SGD.
- Drop out some weights during training (p=0.2-0.5)

**Loss Function**

<img src='images/loss_function.png' width=600>